# EdU S-phase patterns: Ripley's K and pair correlation g(r)

Compares how EdU-labelled replicating DNA is spatially organised across the S-phase stages S1–S4. Each CSV in
`meng_data/` is one field of view, and its stage comes from the file name (`PHASE_PATTERN`: `S2_EduOnly_009.csv` is
S2). Every whole nucleus in a field is one cell of that stage: step 2 finds them and splits nuclei that touch. More
files, or fields with several nuclei, give more replicate cells. The analysis uses the second-order statistics in
`hmt.measure.measure_pair_correlation`:

- **Ripley's K(r)**: mean number of neighbours within r, divided by the density. Complete spatial randomness (CSR) gives K = πr².
  Shown mainly as **L(r) − r** with L = √(K/π): > 0 means clustering at scale r, < 0 regularity.
- **Pair correlation g(r)**: density of pairs at distance r relative to CSR. g = 1 is random, g > 1 clustered.

Both are edge-corrected against the irregular nucleus mask. Both are also unchanged, in expectation, when
localizations are randomly thinned, so cells with different EdU densities can be compared, and the
analysis can run on random subsamples.

How this differs from the histone pipeline (`full_pipeline.ipynb`):

- **One channel, 2D.** These are ThunderSTORM exports with no z. The `hmt_v3` mask functions take two channels,
  so the second is an empty table, as in `single_channel_nanodomains.ipynb`.
- **No DBSCAN calibration.** K and g(r) have no cluster parameters to tune.
- **Quality filter and blink merging come first.** A dSTORM dye stays on for several frames,
  so one EdU site gives a burst of localizations within ~50 nm. In these files about a third of
  localizations have a partner within 50 nm in the very next frame, against about 1% at a 1000-frame lag.
  Left in, the bursts put a large artificial peak in g(r) at short range. `hmt_v3` has no blink merging,
  so it's defined in step 1 below.
- **Drift-correct the CSVs in ThunderSTORM first.** The raw exports drift by ~75–110 nm over the 20,000
  frames, which blurs foci and lowers and widens the short-range g(r) peak. This notebook doesn't correct it.

Steps:

1. Load, quality-filter, merge blinks
2. Nuclei: `preprocess.auto_mask_threshold` and `binarize_nucleus` threshold the field, a watershed splits
   nuclei that touch, and each whole nucleus gets its own mask (`create_radial_contours`)
3. K, L and g(r) (`measure.measure_pair_correlation`) on each of `N_SUBSAMPLES` random subsamples of every
   nucleus, each a fresh `SUBSAMPLE_FRACTION` of its localizations, with a CSR reference: uniform points in the
   same mask (`simulate.add_noise_locs`)
4. Compare S phases: each cell's curves, phase means over cells, summary numbers (including
   `measure.estimate_n_domains_from_g`) and tests across phases
5. Save
6. Nanodomain size from g(r), without clustering: a two-scale Gaussian cluster model fitted to g(r), checked
   on simulated domains of known size, and compared across phases the same way

**Two kinds of interval.**
- *Within a cell.* Every curve and parameter is computed on each subsample. A cell's result is the mean over its
  subsamples, with a `CI_LEVEL` (95%) percentile interval. That interval shows how much the result depends on
  which localizations were drawn: it's the precision of that one cell's measurement.
- *Between cells.* A phase's result is the mean over its cells (each cell counts once), with the 95% confidence
  interval of that mean (a t-interval). Only this one includes cell-to-cell variation, and it needs at least
  2 cells in the phase. The tests across phases also use one value per cell.

A phase with one cell has no between-cell interval, and with one cell per stage only the trend test can run (its
smallest possible p with 4 cells is 0.08). Nuclei from the same field share a dish, a day and an acquisition, so
they may be more alike than nuclei from different fields. The tests treat every nucleus as independent.

Run time is roughly 1 minute per cell with `N_SUBSAMPLES = 20` and 2 minutes with 50, on 32 cores. Step 3 takes
most of it: per cell, 2 grids × (`N_SUBSAMPLES` + `N_CSR`) `measure_pair_correlation` calls of about 5 s each,
run `N_JOBS` at a time in parallel threads. `SHORT["n"]` / `LONG["n"]` set the cost per call, not
`SUBSAMPLE_FRACTION`. Every cell's localizations stay in memory, about 30 MB per million after merging. Open the
notebook from anywhere inside the repo; the setup cell finds the repo root.

In [ ]:
from pathlib import Path
import itertools
import json
import re
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy import ndimage, stats
from scipy.optimize import curve_fit
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree
from skimage.measure import regionprops
from skimage.morphology import local_maxima, reconstruction
from skimage.segmentation import watershed

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hmt_v3").is_dir())
sys.path.insert(0, str(REPO_ROOT))
import hmt_v3 as hmt

## Settings

In [ ]:
# --- Input ---------------------------------------------------------------------
# One CSV per field of view. Its S phase is PHASE_PATTERN's first group in the file name (case-insensitive,
# upper-cased): "S2_EduOnly_009.csv" -> "S2". Files it doesn't match, phases not in PHASE_ORDER and excluded
# files are skipped and listed when this cell runs. A file's id is its name without ".csv".
# Every whole nucleus in a file (step 2) is one cell, with the file's phase. Its id is the file id, or
# "<file id>_n1", "_n2", ... (numbered left to right) when the file has several.
DATA_DIR = REPO_ROOT / "meng_data"
PHASE_PATTERN = r"^(S\d+)_"
PHASE_ORDER = ["S1", "S2", "S3", "S4"]  # plotting order, and the stage order the trend test uses
EXCLUDE_CELLS = [                       # file ids (the whole field) or cell ids (one nucleus) to leave out
    # second exports of the files without "_0": same frames and intensities, positions within a few nm
    "S1_EduOnly_012_0", "S2_EduOnly_009_0", "S3_EduOnly_006_0", "S4_EduOnly_004_0",
]
PHASE_OVERRIDE = {}                     # by cell id, for a nucleus at another stage than its field: {"S2_EduOnly_010_n3": "S3"}
OUTPUT_DIR = REPO_ROOT / "analysis_output_edu_s_phase"

# --- Subsampling and confidence intervals ---------------------------------------------
# Steps 3-6 run on N_SUBSAMPLES random subsamples of each cell, each a fresh SUBSAMPLE_FRACTION of its merged
# localizations (seed RANDOM_SEED + i; 1.0 = all of them every time). Every curve and parameter is computed on
# each subsample and reported as the mean with a CI_LEVEL percentile interval.
# Subsampling comes after blink merging, which needs every frame: thinning first would split blink bursts into
# pieces the merge can't rejoin. K, L and g(r) are unchanged in expectation under random thinning. Each
# subsample's K / g(r) is measured on SHORT["n"] / LONG["n"] of its points, so those n set how wide the
# intervals are, unless a subsample has fewer points.
# The first subsample also builds the nucleus masks (step 2). The mask is not thinning-invariant: below ~0.1 the
# sparse S4 nucleus no longer closes into one region, so check the step 2 masks if you go lower.
SUBSAMPLE_FRACTION = 0.4
N_SUBSAMPLES = 50
CI_LEVEL = 0.95
RANDOM_SEED = 0

# --- 1. Localization quality ------------------------------------------------------
SIGMA_RANGE_NM = (80.0, 250.0)  # fitted PSF width; in-focus single molecules peak at ~140 nm
MAX_UNCERTAINTY_NM = 30.0

# --- 1. Blink merging ----------------------------------------------------------------
MERGE = True
MERGE_RADIUS_NM = 50.0          # max distance between consecutive localizations of one burst
MERGE_MAX_OFF_FRAMES = 1        # dark frames allowed inside a burst

# --- 2. Nucleus masks ------------------------------------------------------------------
MASK_BIN_NM = 50
MASK_SIGMA = 12.0               # smoothing in mask pixels (600 nm); the histone default of 4 leaves gaps
                                # between the sparse late-S foci, so the S3 / S4 masks lose part of the nucleus
MASK_AUTO_FACTOR = 0.5          # threshold = this x Otsu threshold of the smoothed density map
MASK_OVERRIDES = {              # per file id, if the auto mask is off: any of "sigma", "factor", "thresh"
    "S4_EduOnly_003": {"sigma": 30.0},  # late-S nuclei with mostly peripheral foci: 600 nm leaves open rings
}

# --- 2. Whole nuclei -------------------------------------------------------------------
SPLIT_DEPTH_UM = 2.0            # touching nuclei are split where the mask narrows: each needs its own peak in the
                                # distance to the mask edge, at least this far above the neck. 1 um cut
                                # S1_EduOnly_012 in two; 3 um joined two of the four S2_EduOnly_010 nuclei
MIN_NUCLEUS_AREA_UM2 = 50.0     # smaller pieces are debris or fragments; the whole nuclei here are 130-370 um^2
EDGE_MARGIN_UM = 4.0            # closer to the edge of the field counts as cut off, by the frame or by the dim
                                # illumination near the corners; the whole nuclei here are >= 6.8 um from it
MIN_SOLIDITY = 0.9              # area / convex hull area: whole nuclei here are >= 0.95, open or cut-off masks lower
MIN_NUCLEUS_LOCS = 20_000       # merged localizations inside the nucleus

# --- 3. Pair correlation ---------------------------------------------------------------
# Two radial grids: fine bins at short range (single foci) and coarse bins out to the multi-focus /
# nuclear scale. measure_pair_correlation costs ~ n^2 * r_max^3 / dr, so each grid draws its own
# number of points per call (n). Those draws, not SUBSAMPLE_FRACTION, set the run time.
SHORT = dict(r_max=500.0, dr=10.0, n=50_000)
LONG = dict(r_max=3000.0, dr=50.0, n=20_000)
N_CSR = 3                       # uniform-in-mask (CSR) realisations per cell and grid
N_JOBS = -1                     # measure_pair_correlation calls run in parallel threads (joblib); -1 = all cores
BLINK_R_NM = 50.0               # g(r) below this is dominated by residual re-blinking
FOCUS_R_NM = 150.0              # summary "focus-scale g(r)" = mean g over BLINK_R_NM..FOCUS_R_NM
CLUSTER_R_NM = 500.0            # estimate_n_domains_from_g integration limit: the focus-scale excess has
                                # decayed by here in the first four cells checked; beyond it g(r) sits at ~1.02-1.05
                                # and its first crossing of 1 is set by noise

# --- 6. Nanodomain size from g(r) ----------------------------------------------------------
SIZE_FIT_R_NM = 1000.0          # fit g(r) over BLINK_R_NM..this; on these cells 2000 changed sizes by < 5 nm
SIZE_CHECK_SIGMAS_NM = (40.0, 60.0, 90.0)  # known domain sizes (Gaussian SD) for the recovery check
SIZE_CHECK_GROUP_SIGMA_NM = 150.0          # spread of domain groups in the check simulation

# --- Plotting ------------------------------------------------------------------------
# S1 -> S4 is an ordered sequence, so it gets an ordinal one-hue ramp (light -> dark blue), one step per
# PHASE_ORDER entry: lightness falls monotonically, adjacent steps differ by >= 0.06 OKLCH L, lightest is >= 2:1 on white
PHASE_COLORS = dict(zip(PHASE_ORDER, ["#6da7ec", "#2a78d6", "#1c5cab", "#0d366b"]))
CSR_COLOR = "#898781"
INK, INK_2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

# --- Files found in DATA_DIR ------------------------------------------------------------
FILES, file_phase, skipped = {}, {}, []  # file id -> path / S phase
for path in sorted(DATA_DIR.glob("*.csv")):
    match = re.match(PHASE_PATTERN, path.name, re.IGNORECASE)
    phase = match.group(1).upper() if match else None
    if phase in PHASE_ORDER and path.stem not in EXCLUDE_CELLS:
        FILES[path.stem], file_phase[path.stem] = path, phase
    else:
        skipped.append(path.name)
if not FILES:
    raise FileNotFoundError(f"no CSV in {DATA_DIR} has a PHASE_ORDER phase matching {PHASE_PATTERN!r}")
FILES = dict(sorted(FILES.items(), key=lambda item: (PHASE_ORDER.index(file_phase[item[0]]), item[0])))
assert set(PHASE_ORDER) <= set(PHASE_COLORS), "add a PHASE_COLORS step for every PHASE_ORDER entry"
assert set(PHASE_OVERRIDE.values()) <= set(PHASE_ORDER), "PHASE_OVERRIDE phases must be in PHASE_ORDER"

# One acquisition exported twice (e.g. before and after drift correction, which only moves x / y) has the same
# frames and intensities. Analysing both copies would count its nuclei twice.
copies = {}
for file, path in FILES.items():
    head = pd.read_csv(path, usecols=["frame", "intensity [photon]"], nrows=2000).to_numpy()
    copies.setdefault(head.tobytes(), []).append(file)
duplicates = [files for files in copies.values() if len(files) > 1]
if duplicates:
    raise ValueError(f"the same acquisition is in several files: {duplicates}. "
                     f"Add all but one of each to EXCLUDE_CELLS.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for p in PHASE_ORDER:
    in_phase = [f for f in FILES if file_phase[f] == p]
    print(f"{p}: {len(in_phase)} file(s): {', '.join(in_phase)}")
if skipped:
    print(f"skipped (no phase in the name, phase not in PHASE_ORDER, or excluded): {', '.join(skipped)}")

## 1. Load, filter, merge blinks, subsample

For each file (already drift-corrected in ThunderSTORM):

1. **Quality filter.** Keep fits with an in-focus PSF width (`SIGMA_RANGE_NM`) and
   uncertainty < `MAX_UNCERTAINTY_NM`. This drops a bright (~10⁵–10⁶ photon), very wide (sigma > 500 nm)
   population of out-of-focus or aggregate fits. `hmt.preprocess.filter_quantile` isn't used here: it trims
   both tails equally, and the bad fits sit in only one tail.
2. **Blink merging** (`merge_blinks`). Localizations within `MERGE_RADIUS_NM`, in frames no more than
   `MERGE_MAX_OFF_FRAMES + 1` apart, are joined into one burst, placed at the precision-weighted mean.
   `repeat_lag1` / `repeat_lag1000` in the table below are the fractions of localizations with a partner
   within `MERGE_RADIUS_NM` 1 frame and 1000 frames later. At lag 1000 the same dye is rarely still on,
   so that value is the chance level, and `repeat_lag1` far above it shows what the merge removes.
   Dyes that come back after a longer dark period aren't merged, so g(r) below `BLINK_R_NM` still
   carries some of that self-clustering.
3. **Subsample.** A first random `SUBSAMPLE_FRACTION` of the merged localizations (seed `RANDOM_SEED`) builds the
   file's nucleus masks in step 2. Step 3 then draws `N_SUBSAMPLES` fresh subsamples from the merged localizations
   inside each nucleus.

In [ ]:
def partner_fraction(df, radius_nm, lag, every=10):
    """
    Fraction of localizations that have another localization within radius_nm exactly `lag`
    frames later (checked on every `every`-th frame). At lag 1 this counts blink repeats; at a
    long lag, where the same dye is rarely still on, it is the chance level set by local density.
    """
    frame = df["frame"].to_numpy().astype(int)
    xy = df[["x [nm]", "y [nm]"]].to_numpy(float)
    order = np.argsort(frame, kind="stable")
    frame, xy = frame[order], xy[order]
    start = np.searchsorted(frame, np.arange(frame.max() + 2))  # rows of frame t: start[t]:start[t + 1]
    hits = total = 0
    for t in range(frame.min(), frame.max() - lag + 1, every):
        a = xy[start[t]:start[t + 1]]
        b = xy[start[t + lag]:start[t + lag + 1]]
        total += len(a)
        if len(a) and len(b):
            d, _ = cKDTree(b).query(a, distance_upper_bound=radius_nm)
            hits += int(np.count_nonzero(d <= radius_nm))
    return hits / max(total, 1)


def merge_blinks(df, radius_nm=50.0, max_off_frames=1):
    """
    Joins the localizations of one emission burst into a single localization, like ThunderSTORM's
    "merging" post-processing. Two localizations are linked if they are within radius_nm of each
    other and 1 to max_off_frames + 1 frames apart; chains of links form one burst. The merged
    position is the 1/uncertainty^2-weighted mean, its uncertainty the combined one, its intensity
    the sum and its frame the first frame. n_merged = localizations in the burst.
    """
    xyf = df[["x [nm]", "y [nm]", "frame"]].to_numpy(float)
    max_lag = max_off_frames + 1
    # Frames scaled so one Chebyshev-distance search finds |dx|, |dy| <= r and |dframe| <= max_lag
    pts = np.column_stack([xyf[:, :2], xyf[:, 2] * radius_nm / max_lag])
    pairs = cKDTree(pts).query_pairs(radius_nm * (1 + 1e-9), p=np.inf, output_type="ndarray")
    d = xyf[pairs[:, 0]] - xyf[pairs[:, 1]]
    dframe = np.abs(np.rint(d[:, 2]))
    pairs = pairs[(np.hypot(d[:, 0], d[:, 1]) <= radius_nm) & (dframe >= 1) & (dframe <= max_lag)]

    n = len(df)
    links = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(n, n))
    _, burst = connected_components(links, directed=False)

    w = 1.0 / df["uncertainty [nm]"].to_numpy(float) ** 2
    parts = pd.DataFrame({"burst": burst, "w": w, "wx": w * xyf[:, 0], "wy": w * xyf[:, 1],
                          "frame": xyf[:, 2], "intensity": df["intensity [photon]"].to_numpy(float)})
    agg = parts.groupby("burst").agg(w=("w", "sum"), wx=("wx", "sum"), wy=("wy", "sum"), frame=("frame", "min"),
                                     intensity=("intensity", "sum"), n_merged=("w", "size"))
    return pd.DataFrame({"frame": agg["frame"].to_numpy(),
                         "x [nm]": (agg["wx"] / agg["w"]).to_numpy(),
                         "y [nm]": (agg["wy"] / agg["w"]).to_numpy(),
                         "intensity [photon]": agg["intensity"].to_numpy(),
                         "uncertainty [nm]": (1.0 / np.sqrt(agg["w"])).to_numpy(),
                         "n_merged": agg["n_merged"].to_numpy()})

In [ ]:
t_start = time.time()
merged, mask_locs, file_rows = {}, {}, []  # per file: every merged localization / the first subsample (builds the masks)
for file, path in FILES.items():
    t0 = time.time()
    raw = pd.read_csv(path, usecols=["frame", "x [nm]", "y [nm]", "sigma [nm]", "intensity [photon]",
                                     "uncertainty [nm]"], dtype=np.float32)
    keep = raw["sigma [nm]"].between(*SIGMA_RANGE_NM) & (raw["uncertainty [nm]"] < MAX_UNCERTAINTY_NM)
    locs = raw[keep].reset_index(drop=True)
    row = {"file": file, "phase": file_phase[file], "n_raw": len(raw), "n_quality": len(locs),
           "repeat_lag1": partner_fraction(locs, MERGE_RADIUS_NM, 1),
           "repeat_lag1000": partner_fraction(locs, MERGE_RADIUS_NM, 1000)}
    del raw

    if MERGE:
        locs = merge_blinks(locs, MERGE_RADIUS_NM, MERGE_MAX_OFF_FRAMES)
        row.update(n_merged=len(locs), burst_locs_p99=float(locs["n_merged"].quantile(0.99)),
                   burst_locs_max=int(locs["n_merged"].max()))

    # Later steps only use x, y and uncertainty; float32 keeps many files in memory
    merged[file] = locs[["x [nm]", "y [nm]", "uncertainty [nm]"]].astype(np.float32)
    mask_locs[file] = (merged[file].sample(frac=SUBSAMPLE_FRACTION, random_state=RANDOM_SEED).reset_index(drop=True)
                       if SUBSAMPLE_FRACTION < 1.0 else merged[file])
    row["n_subsample"] = len(mask_locs[file])
    file_rows.append(row)
    print(f"{file} [{file_phase[file]}]: {row['n_raw']:,} raw -> {row['n_quality']:,} quality -> "
          f"{len(merged[file]):,} bursts -> {row['n_subsample']:,} per subsample ({time.time() - t0:.0f} s)")

file_qc = pd.DataFrame(file_rows).set_index("file")
file_qc.round(3)

## 2. Nuclei

The `hmt_v3` mask functions are used as in `single_channel_nanodomains.ipynb`, but every nucleus in the field is
kept, not only the largest:

1. **Threshold.** `auto_mask_threshold` (× `MASK_AUTO_FACTOR`) thresholds the density map smoothed by `MASK_SIGMA`,
   and `binarize_nucleus` fills holes.
2. **Split nuclei that touch.** Where two nuclei touch, the mask narrows into a neck. A watershed on the distance to
   the mask edge gives a peak of that distance its own nucleus if it rises at least `SPLIT_DEPTH_UM` above the neck
   to a higher peak. A round or oval nucleus has one peak, so it isn't cut. The cut follows the neck of the mask, not
   the dark gap between the nuclei, so a few hundred nm of one nucleus can end up on its neighbour's side.
3. **Keep whole nuclei.** A candidate is rejected if it is smaller than `MIN_NUCLEUS_AREA_UM2` (debris, fragments),
   within `EDGE_MARGIN_UM` of the edge of the field (cut off by the frame, or by the dim illumination near the
   corners), less convex than `MIN_SOLIDITY` (cut off, or a mask that missed part of the nucleus), or has fewer than
   `MIN_NUCLEUS_LOCS` localizations.

Every whole nucleus is one cell, with the file's S phase. Its id is the file id, or `<file id>_n1`, `_n2`, …
numbered left to right when the file has several. A neighbouring nucleus can be at a different stage from the one
the field was named for. If one is, set its phase with `PHASE_OVERRIDE` or leave it out with `EXCLUDE_CELLS`.

One panel per file, in phase order. Kept nuclei are outlined in cyan and numbered; rejected candidates of at least
`MIN_NUCLEUS_AREA_UM2` are dashed grey, labelled with the reason. Check that each cyan outline follows one nuclear
edge:

- It takes in cytoplasm: raise that file's threshold with `MASK_OVERRIDES` (`"thresh"`, or a higher `"factor"`).
- It cuts into the nucleus: lower it.
- A late-S nucleus with mostly peripheral foci is an open ring: smooth more (`"sigma"`).
- Two touching nuclei stay joined: lower `SPLIT_DEPTH_UM`. One nucleus is cut in two: raise it.

Each cell's subsamples (step 3) are drawn from the merged localizations inside its own mask
(`preprocess.assign_contour_bands`). The localization counts and densities below are for all of those
localizations, not one subsample. The candidate table is saved as `nuclei.csv`.

`MASK_SIGMA` is 12 pixels (600 nm) rather than the histone default of 4. Late-S EdU is a few large foci with
sparse signal between them, and with less smoothing the S3 / S4 masks lose part of the nucleus.

`binarize_nucleus` fills holes, so nucleoli count as inside the nucleus. K, L and g(r) are measured relative
to that whole area, so a large EdU-free nucleolus shows up as clustering at the µm scale.

In [ ]:
def find_nuclei(binary_mask, x_min, y_min):
    """
    Splits a thresholded mask into candidate nuclei and flags the ones that aren't whole. Touching nuclei are
    separated by a watershed on the distance to the mask edge. Each peak of that distance at least SPLIT_DEPTH_UM
    above the neck to a higher peak seeds one nucleus. The seeds are the plateaus of the h-reconstruction, not
    skimage's h_maxima pixels, which can return two equal top pixels of one peak as two seeds. Pieces under
    MIN_NUCLEUS_AREA_UM2 left by the split rejoin the neighbour they share the most edge with.
    Returns the label image (0 = outside) and one row per candidate: area, solidity (area / convex hull area),
    centre, distance to the edge of the field, and why it isn't a whole nucleus ("" if it is).
    """
    dist = ndimage.distance_transform_edt(binary_mask) * MASK_BIN_NM / 1e3  # µm
    seeds, _ = ndimage.label(local_maxima(reconstruction(dist - SPLIT_DEPTH_UM, dist)) & binary_mask)
    labels = watershed(-dist, seeds, mask=binary_mask)
    px_um2 = (MASK_BIN_NM / 1e3) ** 2
    for k in np.argsort(np.bincount(labels.ravel())[1:]) + 1:  # smallest piece first
        piece = labels == k
        if 0 < piece.sum() * px_um2 < MIN_NUCLEUS_AREA_UM2:
            ring = labels[ndimage.binary_dilation(piece) & ~piece]
            ring = ring[ring > 0]
            if ring.size:
                labels[piece] = np.bincount(ring).argmax()
    rows = []
    for region in regionprops(labels):
        x0, y0, x1, y1 = region.bbox
        edge_um = min(x0, y0, labels.shape[0] - x1, labels.shape[1] - y1) * MASK_BIN_NM / 1e3
        area_um2 = region.area * px_um2
        reason = ("small" if area_um2 < MIN_NUCLEUS_AREA_UM2 else "at the edge" if edge_um < EDGE_MARGIN_UM
                  else "irregular" if region.solidity < MIN_SOLIDITY else "")
        rows.append({"label": region.label, "area_um2": area_um2, "solidity": region.solidity, "edge_um": edge_um,
                     "x_um": (x_min + (region.centroid[0] + 0.5) * MASK_BIN_NM) / 1e3,
                     "y_um": (y_min + (region.centroid[1] + 0.5) * MASK_BIN_NM) / 1e3, "reason": reason})
    return labels, pd.DataFrame(rows, columns=["label", "area_um2", "solidity", "edge_um", "x_um", "y_um", "reason"])


masks, nuclei, file_of, phase_of, cell_rows, candidate_tables = {}, {}, {}, {}, [], []
n_cols = min(4, len(mask_locs))
n_rows = -(-len(mask_locs) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.6 * n_cols, 4.9 * n_rows), squeeze=False)
for ax in axes.flat[len(mask_locs):]:
    ax.axis("off")
for ax, (file, locs) in zip(axes.flat, mask_locs.items()):
    opts = {"sigma": MASK_SIGMA, "factor": MASK_AUTO_FACTOR, **MASK_OVERRIDES.get(file, {})}
    empty = locs.iloc[:0]
    thresh = opts["thresh"] if "thresh" in opts else hmt.preprocess.auto_mask_threshold(
        locs, empty, bin_size=MASK_BIN_NM, sigma=opts["sigma"], factor=opts["factor"])
    binary_mask, _, _ = hmt.preprocess.binarize_nucleus(  # num_nuclei: keep every region, find_nuclei sorts them out
        locs, empty, thresh=thresh, bin_size=MASK_BIN_NM, sigma=opts["sigma"], num_nuclei=10**9)
    # Mask grid origin: min x/y of the PRE-mask table (every later step that maps locs onto the mask needs it)
    x_min, y_min = hmt.preprocess.mask_origin(locs)
    labels, cands = find_nuclei(binary_mask, x_min, y_min)
    cands.insert(0, "file", file)
    cands["cell"], cands["n_locs"] = "", np.nan

    whole = cands.index[cands["reason"] == ""]
    whole = whole[np.argsort(cands.loc[whole, "x_um"].to_numpy())]  # numbered left to right
    for k, i in enumerate(whole, 1):
        cell = file if len(whole) == 1 else f"{file}_n{k}"
        nucleus_mask = labels == cands.at[i, "label"]
        _, contour_bands = hmt.preprocess.create_radial_contours(nucleus_mask)
        # Every merged localization inside the nucleus (band -1 = outside): the pool step 3 draws its subsamples from
        band = hmt.preprocess.assign_contour_bands(merged[file][["x [nm]", "y [nm]"]], contour_bands, MASK_BIN_NM,
                                                   x_min, y_min)["contour_band"].to_numpy()
        nucleus = merged[file][band >= 0].reset_index(drop=True)
        cands.loc[i, ["cell", "n_locs"]] = [cell, len(nucleus)]
        if len(nucleus) < MIN_NUCLEUS_LOCS:
            cands.at[i, "reason"] = "few localizations"
        elif cell in EXCLUDE_CELLS:
            cands.at[i, "reason"] = "excluded"
        else:
            area_nm2 = float(nucleus_mask.sum()) * MASK_BIN_NM ** 2
            masks[cell] = dict(mask=nucleus_mask, bands=contour_bands, x_min=x_min, y_min=y_min, area_nm2=area_nm2)
            nuclei[cell], file_of[cell] = nucleus, file
            phase_of[cell] = PHASE_OVERRIDE.get(cell, file_phase[file])
            cell_rows.append({"cell": cell, "phase": phase_of[cell], "file": file, "mask_thresh": thresh,
                              "mask_area_um2": area_nm2 / 1e6, "solidity": cands.at[i, "solidity"],
                              "edge_um": cands.at[i, "edge_um"], "n_nucleus": len(nucleus),
                              "density_per_um2": len(nucleus) / area_nm2 * 1e6})
    candidate_tables.append(cands)

    x_edges = x_min + np.arange(binary_mask.shape[0] + 1) * MASK_BIN_NM
    y_edges = y_min + np.arange(binary_mask.shape[1] + 1) * MASK_BIN_NM
    density, _, _ = np.histogram2d(locs["x [nm]"], locs["y [nm]"], bins=(x_edges, y_edges))
    ax.imshow(np.log1p(density.T), origin="lower", cmap="magma",
              extent=[x_edges[0] / 1e3, x_edges[-1] / 1e3, y_edges[0] / 1e3, y_edges[-1] / 1e3])
    x_px, y_px = (x_edges[:-1] + MASK_BIN_NM / 2) / 1e3, (y_edges[:-1] + MASK_BIN_NM / 2) / 1e3
    for _, c in cands[cands["reason"] != "small"].iterrows():
        kept = c["reason"] == ""
        ax.contour(x_px, y_px, (labels == c["label"]).T.astype(float), levels=[0.5],
                   colors="#3fd0ff" if kept else "#c3c2b7", linewidths=1.2 if kept else 0.8,
                   linestyles="solid" if kept else "dashed")
        ax.text(c["x_um"], c["y_um"], c["cell"][len(file) + 1:] if kept else c["reason"], ha="center", va="center",
                color="#3fd0ff" if kept else "#c3c2b7", fontsize=9 if kept else 8)
    n_kept = int((cands["reason"] == "").sum())
    ax.set_title(f"{file} [{file_phase[file]}]\n{n_kept} whole nucle{'us' if n_kept == 1 else 'i'}, "
                 f"threshold {thresh:.2f}", loc="left", fontsize=10)
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "masks.png", dpi=200)
plt.show()

candidates = pd.concat(candidate_tables, ignore_index=True)
if not nuclei:
    raise RuntimeError("no whole nucleus found: check the masks above and the step 2 settings")
order = sorted(nuclei, key=lambda c: (PHASE_ORDER.index(phase_of[c]), c))
nuclei, masks = {c: nuclei[c] for c in order}, {c: masks[c] for c in order}
phases = [p for p in PHASE_ORDER if p in phase_of.values()]
cells_in = {p: [c for c in nuclei if phase_of[c] == p] for p in phases}
qc = pd.DataFrame(cell_rows).set_index("cell").loc[order].join(file_qc.drop(columns="phase"), on="file")
for p in phases:
    print(f"{p}: {len(cells_in[p])} cell(s): {', '.join(cells_in[p])}")
print("Candidates of at least MIN_NUCLEUS_AREA_UM2 (reason empty = kept):")
display(candidates.loc[candidates["reason"] != "small",
                       ["file", "cell", "area_um2", "solidity", "edge_um", "n_locs", "reason"]].round(2))
qc[["phase", "file", "mask_thresh", "mask_area_um2", "solidity", "n_nucleus", "density_per_um2"]].round(2)

## 3. Ripley's K, L(r) and g(r)

For each cell, `N_SUBSAMPLES` subsamples are drawn, each a fresh random `SUBSAMPLE_FRACTION` of the merged
localizations inside the mask. For each radial grid (`SHORT`, `LONG`), `n` points of every subsample go through
`hmt.measure.measure_pair_correlation`, `N_JOBS` calls at a time. It computes a translation edge-corrected K(r)
inside the mask, L = √(K/π), and g(r) = K′(r) / 2πr. Curves are the mean over the subsamples; bands are the
pointwise `CI_LEVEL` (95%) percentile interval.

**CSR reference.** `hmt.simulate.add_noise_locs` places the same number of points uniformly inside the same
mask, and they go through the same estimator. Their g ≈ 1 and L − r ≈ 0 show the edge correction holds. The
spread of those curves is what a random pattern with this many points gives.

In [ ]:
def pcf_draws(draw, grid, n_draws, mk):
    # measure_pair_correlation on the point sets draw(0) .. draw(n_draws - 1), N_JOBS at a time; returns r and
    # stacked K, L, g. The sets are drawn first, in order, so seeded draws don't depend on thread timing.
    sets = [draw(i)[["x [nm]", "y [nm]"]] for i in range(n_draws)]
    curves = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(hmt.measure.measure_pair_correlation)(pts, mk["mask"], mk["x_min"], mk["y_min"], r_max=grid["r_max"],
                                                      dr=grid["dr"], px_size=MASK_BIN_NM) for pts in sets)
    return {"r": curves[0]["r"], **{k: np.array([c[k] for c in curves]) for k in ("K", "L", "g")}}


def subsample(nucleus, i):
    # Subsample i: a random SUBSAMPLE_FRACTION of the cell's merged localizations inside the mask
    return nucleus.sample(frac=SUBSAMPLE_FRACTION, random_state=RANDOM_SEED + i) if SUBSAMPLE_FRACTION < 1.0 else nucleus


np.random.seed(RANDOM_SEED)  # add_noise_locs draws from NumPy's global random state
GRIDS = {"short": SHORT, "long": LONG}
results = {}  # (cell, grid) -> {"data": pcf_draws(...), "csr": pcf_draws(...), "n": points per draw}
for k, (cell, nucleus) in enumerate(nuclei.items(), 1):
    mk = masks[cell]
    t0 = time.time()
    for grid_name, grid in GRIDS.items():
        n = min(grid["n"], int(SUBSAMPLE_FRACTION * len(nucleus)))
        # row i of both grids' results comes from subsample i, so step 4 can join them into one curve
        data = pcf_draws(lambda i: subsample(nucleus, i).sample(n, random_state=RANDOM_SEED + i), grid, N_SUBSAMPLES, mk)
        csr = pcf_draws(lambda i: hmt.simulate.add_noise_locs(n, mk["bands"], np.zeros(1), mk["x_min"], mk["y_min"],
                                                              px_size=MASK_BIN_NM), grid, N_CSR, mk)
        results[(cell, grid_name)] = {"data": data, "csr": csr, "n": n}
    print(f"[{k}/{len(nuclei)}] {cell}: {N_SUBSAMPLES} subsamples + {N_CSR} CSR per grid, "
          f"{results[(cell, 'short')]['n']:,} / {results[(cell, 'long')]['n']:,} points, {time.time() - t0:.0f} s")

## 4. Compare S phases

The curves come in two views:

- **Each cell** (first figure): one panel per phase, one line per cell, with that cell's 95% interval over its
  subsamples. Use it to spot an unusual cell before it's pooled with the others.
- **Each phase** (next two figures): the mean of the phase's cells, with the 95% confidence interval of that mean
  across cells. The legend gives the number of cells. A phase with one cell has no interval, so it's a bare line.

How to read the curves:

- **g(r)** is the pair density at distance r relative to random. At short range its height measures how
  strongly EdU concentrates into foci at that scale. The distance where it falls back toward 1 is roughly how
  far a cluster extends. A long tail above 1 is clustering of foci into larger groups, or nucleus-scale
  heterogeneity such as a peripheral or perinucleolar pattern.
- **L(r) − r** is cumulative: it adds up the excess neighbours out to r. Its maximum sits at a larger scale
  than the g(r) peak, roughly the size of the dominant clusters.
- The grey "CSR range" is the range of every cell's CSR curves.
- The grey region below `BLINK_R_NM` is dominated by dyes that re-blink after a long dark period. Don't
  interpret g(r) there.

In [ ]:
def style(ax, xlabel, ylabel, title):
    ax.set_xlabel(xlabel, color=INK_2)
    ax.set_ylabel(ylabel, color=INK_2)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelcolor=INK_2)
    ax.grid(True, color=GRID, lw=0.6)
    ax.set_axisbelow(True)


def label_lines(ax, points, min_gap_px=13):
    # Direct labels just right of the given (x, y) point on each line, nudged apart vertically so they
    # don't overlap. Pick points where the lines are well separated. Call after tight_layout.
    disp = {k: ax.transData.transform(v)[1] for k, v in points.items()}
    placed = []
    for k in sorted(disp, key=disp.get):
        y = disp[k] if not placed else max(disp[k], placed[-1] + min_gap_px)
        placed.append(y)
        ax.annotate(k, xy=points[k], xytext=(5, (y - disp[k]) * 72 / ax.figure.dpi), textcoords="offset points",
                    va="bottom", ha="left", color=INK_2, fontsize=9, annotation_clip=False)


def mean_ci(a):
    # mean over a cell's subsamples (rows) and the pointwise CI_LEVEL percentile interval
    q = (1 - CI_LEVEL) / 2
    return a.mean(0), np.quantile(a, q, axis=0), np.quantile(a, 1 - q, axis=0)


def across_cells(a):
    # mean over cells (rows) and the CI_LEVEL t-interval of that mean; NaN interval for a single cell
    a = np.asarray(a, float)
    m = a.mean(0)
    if len(a) < 2:
        return m, np.full_like(m, np.nan), np.full_like(m, np.nan)
    half = stats.t.ppf(0.5 + CI_LEVEL / 2, len(a) - 1) * a.std(0, ddof=1) / np.sqrt(len(a))
    return m, m - half, m + half


def phase_curve(phase, grid_name, key):
    # A phase's curve: the mean over its cells of each cell's subsample-mean curve, with the interval across cells
    return across_cells([results[(c, grid_name)]["data"][key].mean(0) for c in cells_in[phase]])


def csr_band(grid_name, key, transform=lambda r, v: v):
    # min / max over every cell's CSR curves (they share the same radii)
    curves = [transform(res["csr"]["r"], res["csr"][key]) for (c, g), res in results.items() if g == grid_name]
    stacked = np.vstack(curves)
    return stacked.min(0), stacked.max(0)


first = next(iter(nuclei))
r_long = results[(first, "long")]["data"]["r"]

# --- Each cell: short-range g(r) and L(r) - r, one panel per phase ---
r_short = results[(first, "short")]["data"]["r"] - SHORT["dr"] / 2  # g(r) is a finite difference of K: bin centres
fig, axes = plt.subplots(2, len(phases), figsize=(3.6 * len(phases), 7.0), sharey="row", squeeze=False)
for j, phase in enumerate(phases):
    c = PHASE_COLORS[phase]
    for cell in cells_in[phase]:
        m, lo, hi = mean_ci(results[(cell, "short")]["data"]["g"])
        axes[0, j].fill_between(r_short, lo, hi, color=c, alpha=0.25, lw=0)
        axes[0, j].plot(r_short, m, color=c, lw=1.2)
        m, lo, hi = mean_ci(results[(cell, "long")]["data"]["L"] - r_long)
        axes[1, j].fill_between(r_long, lo, hi, color=c, alpha=0.25, lw=0)
        axes[1, j].plot(r_long, m, color=c, lw=1.2)
    n = len(cells_in[phase])
    axes[0, j].axhline(1, color=AXIS, lw=1)
    axes[0, j].axvspan(0, BLINK_R_NM, color="#f0efec", zorder=0)
    axes[1, j].axhline(0, color=AXIS, lw=1)
    style(axes[0, j], "r (nm)", "g(r)" if j == 0 else "", f"{phase}: {n} cell{'s' if n > 1 else ''}, g(r)")
    style(axes[1, j], "r (nm)", "L(r) − r (nm)" if j == 0 else "", f"{phase}: L(r) − r")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "per_cell_curves.png", dpi=200)
plt.show()

# --- Each phase: Ripley's K and L(r) - r (long grid) ---
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
ends_L = {}
for phase in phases:
    label = f"{phase} (n = {len(cells_in[phase])})"
    K_mean, _, _ = phase_curve(phase, "long", "K")
    L_mean, L_lo, L_hi = phase_curve(phase, "long", "L")
    c = PHASE_COLORS[phase]
    axes[0].plot(r_long, K_mean / 1e6, color=c, lw=1.8, label=label)
    axes[1].fill_between(r_long, L_lo - r_long, L_hi - r_long, color=c, alpha=0.2, lw=0)
    axes[1].plot(r_long, L_mean - r_long, color=c, lw=1.8, label=label)
    ends_L[phase] = (r_long[-1], L_mean[-1] - r_long[-1])
axes[0].plot(r_long, np.pi * r_long ** 2 / 1e6, color=CSR_COLOR, ls="--", lw=1.2, label="CSR (πr²)")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlim(r_long[0], r_long[-1] * 1.1)
style(axes[0], "r (nm)", "K(r) (µm²)", "Ripley's K")
lo, hi = csr_band("long", "L", lambda rr, v: v - rr)
axes[1].fill_between(r_long, lo, hi, color=CSR_COLOR, alpha=0.3, lw=0, label="CSR range")
axes[1].axhline(0, color=AXIS, lw=1)
axes[1].set_xlim(0, r_long[-1] * 1.08)
style(axes[1], "r (nm)", "L(r) − r (nm)", "Ripley's L(r) − r")
axes[0].legend(frameon=False, loc="upper left")
axes[1].legend(frameon=False, loc="upper right")
fig.tight_layout()
label_lines(axes[1], ends_L)  # K converges onto πr² at large r, so it relies on the legend
fig.savefig(OUTPUT_DIR / "ripley_K_L.png", dpi=200)
plt.show()

# --- Each phase: g(r), short range (fine bins) and long range ---
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
label_points = {}  # short-range panel only: the long-range curves overlap except in their steep first bins
for ax, grid_name in zip(axes, ("short", "long")):
    dr = GRIDS[grid_name]["dr"]
    rc = results[(first, grid_name)]["data"]["r"] - dr / 2  # g(r) is a finite difference of K: bin centres
    for phase in phases:
        g_mean, g_lo, g_hi = phase_curve(phase, grid_name, "g")
        c = PHASE_COLORS[phase]
        ax.fill_between(rc, g_lo, g_hi, color=c, alpha=0.2, lw=0)
        ax.plot(rc, g_mean, color=c, lw=1.8, label=f"{phase} (n = {len(cells_in[phase])})")
        if grid_name == "short":
            label_points[phase] = (FOCUS_R_NM, float(np.interp(FOCUS_R_NM, rc, g_mean)))
    lo, hi = csr_band(grid_name, "g")
    ax.fill_between(rc, lo, hi, color=CSR_COLOR, alpha=0.3, lw=0, label="CSR range")
    ax.axhline(1, color=AXIS, lw=1)
    ax.axvspan(0, BLINK_R_NM, color="#f0efec", zorder=0)
    ax.set_xlim(0, GRIDS[grid_name]["r_max"] * 1.08)
    if grid_name == "short":
        style(ax, "r (nm)", "g(r)", f"Pair correlation, 0–{SHORT['r_max']:.0f} nm ({dr:.0f} nm bins)")
    else:
        style(ax, "r (nm)", "g(r)", f"Pair correlation, 0–{LONG['r_max'] / 1e3:.0f} µm ({dr:.0f} nm bins)")
    ax.text(BLINK_R_NM / 2, 0.02, "re-blinking", transform=ax.get_xaxis_transform(), rotation=90,
            ha="center", va="bottom", color=MUTED, fontsize=8)
    ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
label_lines(axes[0], label_points)
fig.savefig(OUTPUT_DIR / "pair_correlation.png", dpi=200)
plt.show()

### Summary numbers

Each subsample's short-range and long-range curves are joined into one g(r): fine bins up to `SHORT["r_max"]`,
coarse bins beyond. Only r ≥ `BLINK_R_NM` is used. The cell below shows three tables:

1. **Per cell** (`summary`): the mean over the cell's `N_SUBSAMPLES` subsamples and, in brackets, their 95%
   percentile interval (`<column>_lo`, `<column>_hi` in `summary`).
2. **Per phase** (`phase_summary`): each cell counts once, with its subsample mean. The mean over the phase's cells
   and, in brackets, the 95% confidence interval of that mean (t-interval; none with one cell).
3. **Tests across phases** (`phase_tests`), one value per cell:
   - `kruskal_p`: Kruskal–Wallis test over the phases with at least 2 cells. Does any phase differ?
   - `trend_rho`, `trend_p`: Spearman correlation between stage (position in `PHASE_ORDER`) and value over all
     cells, with a permutation p-value (exact up to 7 cells). Does the value rise or fall through S phase? It runs
     with one cell per stage too, but 4 cells can't give p below 2/24 ≈ 0.08.
   - Pairwise (`phase_pairs`, shown once any pair can be tested): Mann–Whitney U and Welch t for each pair of
     phases with at least 2 cells each, as in `drivers/05_compare.py`, plus Holm-corrected p-values within each
     metric. With 3 cells per phase Mann–Whitney can't go below p = 0.1, so with few cells Welch t is the more
     useful one, if the per-cell values are roughly normal.

| Column | What it is |
|---|---|
| `g_focus` | mean g(r) over `BLINK_R_NM`–`FOCUS_R_NM`: how strongly EdU concentrates at the scale of single foci |
| `corr_length_nm` | first r where g(r) − 1 falls below 1/e of its value at `BLINK_R_NM`, a model-free size of the clustering |
| `max_L_minus_r_nm`, `r_at_max_L_nm` | height and position of the L(r) − r peak (long grid), roughly the size of the dominant clusters |
| `cluster_density_per_um2`, `n_clusters` | `hmt.measure.estimate_n_domains_from_g`: a Neyman–Scott model gives cluster density κ = 1 / ∫(g − 1) 2πr dr, integrated from `BLINK_R_NM` to `CLUSTER_R_NM`; `n_clusters` = κ × mask area. It's an *apparent* density: leaving out r < `BLINK_R_NM` inflates it, and clusters that overlap or are themselves grouped break the model's assumptions. Compare it across cells rather than reading it as an absolute focus count. |

`corr_length_nm` and `r_at_max_L_nm` are read off the radial grid (10 and 50 nm bins), so when every subsample
lands in the same bin their interval has zero width.

In the plot, each dot is a cell, with its subsample interval as a thin bar (often hidden by the dot). The black
line and bar beside each phase are the mean over its cells and the 95% confidence interval of that mean. The x
labels give the Kruskal–Wallis and trend tests.

In [ ]:
def join_g(short, long, i):
    # One g(r) curve from draw i: fine bins up to SHORT r_max, coarse bins beyond, at bin centres, r >= BLINK_R_NM
    rs, rl = short["r"] - SHORT["dr"] / 2, long["r"] - LONG["dr"] / 2
    beyond = rl > SHORT["r_max"]
    r = np.concatenate([rs, rl[beyond]])
    g = np.concatenate([short["g"][i], long["g"][i][beyond]])
    ok = r >= BLINK_R_NM
    return r[ok], g[ok]


def curve_metrics(short, long, area_nm2):
    rows = []
    for i in range(min(len(short["g"]), len(long["g"]))):
        r, g = join_g(short, long, i)
        below = np.nonzero(g - 1 <= (g[0] - 1) / np.e)[0]
        H = long["L"][i] - long["r"]
        n_clusters, kappa, _ = hmt.measure.estimate_n_domains_from_g({"r": r, "g": g, "area": area_nm2},
                                                                    r_domain=CLUSTER_R_NM)
        rows.append({"g_focus": g[r <= FOCUS_R_NM].mean(),
                     "corr_length_nm": r[below[0]] if len(below) else np.nan,
                     "max_L_minus_r_nm": H.max(), "r_at_max_L_nm": long["r"][np.argmax(H)],
                     "cluster_density_per_um2": kappa * 1e6, "n_clusters": n_clusters})
    return pd.DataFrame(rows)


def summarise(draws):
    # Per cell: each column's mean over the subsamples, then its CI_LEVEL percentile interval (<column>_lo, _hi)
    q = (1 - CI_LEVEL) / 2
    by_cell = draws.groupby(level="cell", sort=False)
    out = pd.concat([by_cell.mean(), by_cell.quantile(q).add_suffix("_lo"),
                     by_cell.quantile(1 - q).add_suffix("_hi")], axis=1)
    out = out[[c for col in draws.columns for c in (col, f"{col}_lo", f"{col}_hi")]]
    out.insert(0, "phase", out.index.map(phase_of))
    return out


def ci_text(summary, digits):
    # Readable view of a summarise() table: "mean (lo–hi)" for each column in digits, rounded to digits[column]
    return pd.DataFrame({col: [f"{m:.{d}f} ({lo:.{d}f}–{hi:.{d}f})"
                               for m, lo, hi in summary[[col, f"{col}_lo", f"{col}_hi"]].to_numpy()]
                         for col, d in digits.items()}, index=summary.index)


def holm(p):
    # Holm-Bonferroni adjusted p-values; NaN entries are left out of the count
    p = np.asarray(p, float)
    out, ok = np.full_like(p, np.nan), np.isfinite(p)
    order = np.argsort(p[ok])
    m = len(order)
    out[np.flatnonzero(ok)[order]] = np.minimum(np.maximum.accumulate((m - np.arange(m)) * p[ok][order]), 1)
    return out


def spearman(x, y, axis=-1):
    # Spearman rho along axis (vectorised, for permutation_test)
    rx, ry = stats.rankdata(x, axis=axis), stats.rankdata(y, axis=axis)
    rx, ry = rx - rx.mean(axis, keepdims=True), ry - ry.mean(axis, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        return (rx * ry).sum(axis) / np.sqrt((rx ** 2).sum(axis) * (ry ** 2).sum(axis))


def kruskal_p(groups):
    # Kruskal-Wallis p-value over the groups; undefined (NaN) when every value is identical
    try:
        return stats.kruskal(*groups).pvalue
    except ValueError:
        return np.nan


PAIR_COLUMNS = ["metric", "phase_a", "phase_b", "n_a", "n_b", "mannwhitney_p", "welch_t_p"]


def compare_phases(summary, metrics):
    """
    Compares phases with one value per cell (its subsample mean). Returns three tables:
      groups: per metric and phase, the number of cells, their mean and SD, and the CI_LEVEL t-interval of the mean
      tests:  per metric, Kruskal-Wallis over the phases with >= 2 cells, and Spearman rho of value against stage
              (PHASE_ORDER position) over all cells, with a permutation p-value
      pairs:  per metric and pair of phases with >= 2 cells each, Mann-Whitney U and Welch t, plus Holm-corrected p
    """
    groups, tests, pairs = [], [], []
    for metric in metrics:
        values = {p: summary.loc[summary["phase"] == p, metric].dropna().to_numpy(float) for p in phases}
        for p, v in values.items():
            m, lo, hi = (x[0] for x in across_cells(v[:, None])) if len(v) else (np.nan,) * 3
            groups.append({"metric": metric, "phase": p, "n_cells": len(v), "mean": m,
                           "sd": v.std(ddof=1) if len(v) > 1 else np.nan, "ci_lo": lo, "ci_hi": hi})

        testable = [v for v in values.values() if len(v) >= 2]
        stage = np.concatenate([np.full(len(v), PHASE_ORDER.index(p)) for p, v in values.items()])
        y = np.concatenate(list(values.values()))
        varies = len(y) >= 3 and np.ptp(y) > 0 and len(np.unique(stage)) >= 2  # rho is undefined otherwise
        trend = (stats.permutation_test((y,), lambda v, axis: spearman(stage, v, axis), vectorized=True,
                                        permutation_type="pairings", n_resamples=9999,
                                        rng=np.random.default_rng(RANDOM_SEED))
                 if varies else None)
        tests.append({"metric": metric, "n_cells": len(y),
                      "kruskal_p": kruskal_p(testable) if len(testable) >= 2 else np.nan,
                      "trend_rho": trend.statistic if trend else np.nan,
                      "trend_p": trend.pvalue if trend else np.nan})

        rows = []
        for a, b in itertools.combinations(phases, 2):
            va, vb = values[a], values[b]
            enough = len(va) >= 2 and len(vb) >= 2
            rows.append([metric, a, b, len(va), len(vb),
                         stats.mannwhitneyu(va, vb, alternative="two-sided").pvalue if enough else np.nan,
                         stats.ttest_ind(va, vb, equal_var=False).pvalue if enough else np.nan])
        rows = pd.DataFrame(rows, columns=PAIR_COLUMNS)
        for col in ("mannwhitney_p", "welch_t_p"):
            rows[f"{col}_holm"] = holm(rows[col])
        pairs.append(rows)
    return (pd.DataFrame(groups).set_index(["metric", "phase"]), pd.DataFrame(tests).set_index("metric"),
            pd.concat(pairs, ignore_index=True))


def phase_text(groups, digits):
    # Readable view of compare_phases' groups: "mean (lo–hi)" over cells; just the mean for a one-cell phase
    out = {}
    for metric, d in digits.items():
        g = groups.loc[metric]
        out[metric] = [f"{r.mean:.{d}f} ({r.ci_lo:.{d}f}–{r.ci_hi:.{d}f})" if r.n_cells > 1 else f"{r.mean:.{d}f}"
                       for r in g.itertuples()]
    return pd.DataFrame(out, index=g.index).assign(n_cells=g["n_cells"])


def show_pairs(pairs):
    if pairs[["mannwhitney_p", "welch_t_p"]].notna().any().any():
        display(pairs.round(4))
    else:
        print("No pairwise tests: they need at least 2 cells in each of two phases.")


metric_draws = pd.concat({cell: curve_metrics(results[(cell, "short")]["data"], results[(cell, "long")]["data"],
                                              masks[cell]["area_nm2"]) for cell in nuclei},
                         names=["cell", "draw"])
summary = summarise(metric_draws)
metric_digits = {"g_focus": 2, "corr_length_nm": 0, "max_L_minus_r_nm": 1, "r_at_max_L_nm": 0,
                 "cluster_density_per_um2": 2, "n_clusters": 0}
phase_summary, phase_tests, phase_pairs = compare_phases(summary, list(metric_digits))
display(ci_text(summary, metric_digits))
display(phase_text(phase_summary, metric_digits))
display(phase_tests.round(4))
show_pairs(phase_pairs)

In [ ]:
def fmt_p(p):
    return "n/a" if not np.isfinite(p) else "p < 0.0001" if p < 1e-4 else f"p = {p:.2g}"


def phase_panels(summary, groups, tests, panels, width=3.3):
    # One panel per metric. Each cell is a dot with its subsample interval (thin bar). Beside each phase, the mean
    # over its cells (black line) with the CI_LEVEL confidence interval. The x label gives the tests across phases.
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), 3.9))
    x = np.arange(len(phases))
    for ax, (col, title) in zip(np.atleast_1d(axes), panels):
        for xi, phase in zip(x, phases):
            rows = summary[summary["phase"] == phase]
            spread = min(0.12, 0.03 * (len(rows) - 1))
            xs = xi - 0.15 + np.linspace(-spread, spread, len(rows))
            m, lo, hi = (rows[k].to_numpy(float) for k in (col, f"{col}_lo", f"{col}_hi"))
            ax.errorbar(xs, m, yerr=[np.maximum(m - lo, 0), np.maximum(hi - m, 0)], fmt="o", ms=6,
                        color=PHASE_COLORS[phase], elinewidth=1, capsize=0)
            g = groups.loc[(col, phase)]
            yerr = [[g["mean"] - g["ci_lo"]], [g["ci_hi"] - g["mean"]]] if g["n_cells"] > 1 else None
            ax.errorbar(xi + 0.17, g["mean"], yerr=yerr, fmt="_", ms=16, mew=2, color=INK, elinewidth=1.3,
                        capsize=4, capthick=1.3)
        t = tests.loc[col]
        trend = f"trend ρ = {t['trend_rho']:+.2f}, {fmt_p(t['trend_p'])}" if np.isfinite(t["trend_rho"]) else "trend: n/a"
        ax.set_xticks(x, phases)
        ax.set_xlim(-0.6, len(phases) - 0.4)
        ax.margins(y=0.2)
        style(ax, f"Kruskal–Wallis {fmt_p(t['kruskal_p'])}\n{trend}", "", title)
        ax.xaxis.label.set_size(9)
    fig.tight_layout()
    return fig


panels = [("g_focus", f"g(r), {BLINK_R_NM:.0f}–{FOCUS_R_NM:.0f} nm"),
          ("corr_length_nm", "Correlation length (nm)"),
          ("max_L_minus_r_nm", "Peak L(r) − r (nm)"),
          ("r_at_max_L_nm", "r at peak L(r) − r (nm)"),
          ("cluster_density_per_um2", "Apparent cluster density (µm⁻²)")]
fig = phase_panels(summary, phase_summary, phase_tests, panels)
fig.savefig(OUTPUT_DIR / "summary_metrics.png", dpi=200)
plt.show()

## 5. Save

In [ ]:
rows = []
for (cell, grid_name), res in results.items():
    d, c = res["data"], res["csr"]
    for key in ("K", "L", "g"):
        m, lo, hi = mean_ci(d[key])
        rows.append(pd.DataFrame({"cell": cell, "phase": phase_of[cell], "grid": grid_name, "stat": key,
                                  "r_nm": d["r"], "mean": m, "ci_lo": lo, "ci_hi": hi,
                                  "csr_min": c[key].min(0), "csr_max": c[key].max(0)}))
pd.concat(rows, ignore_index=True).to_csv(OUTPUT_DIR / "curves.csv", index=False)  # g(r): the bin (r_nm - dr, r_nm]
rows = []
for phase in phases:
    for grid_name in GRIDS:
        for key in ("K", "L", "g"):
            m, lo, hi = phase_curve(phase, grid_name, key)
            rows.append(pd.DataFrame({"phase": phase, "n_cells": len(cells_in[phase]), "grid": grid_name, "stat": key,
                                      "r_nm": results[(first, grid_name)]["data"]["r"], "mean": m, "ci_lo": lo,
                                      "ci_hi": hi}))
pd.concat(rows, ignore_index=True).to_csv(OUTPUT_DIR / "curves_per_phase.csv", index=False)
metric_draws.to_csv(OUTPUT_DIR / "metrics_per_draw.csv")
qc.join(summary.drop(columns="phase")).to_csv(OUTPUT_DIR / "summary.csv")
phase_summary.to_csv(OUTPUT_DIR / "phase_summary.csv")
phase_tests.to_csv(OUTPUT_DIR / "phase_tests.csv")
phase_pairs.to_csv(OUTPUT_DIR / "phase_pairwise.csv", index=False)
candidates.to_csv(OUTPUT_DIR / "nuclei.csv", index=False)

settings = {k: (str(v) if isinstance(v, Path) else v) for k, v in globals().items()
            if k.isupper() and isinstance(v, (int, float, str, tuple, list, dict, Path))}
settings["FILES"] = {k: str(v) for k, v in FILES.items()}
settings["phase_of"] = phase_of
settings["file_of"] = file_of
with open(OUTPUT_DIR / "settings.json", "w") as f:
    json.dump(settings, f, indent=2, default=str)
print(f"outputs -> {OUTPUT_DIR} | total {time.time() - t_start:.0f} s")

## 6. Nanodomain size from g(r), without clustering

Two localizations in the same nanodomain are rarely more than a domain diameter apart, so the domain's size is
written into the short-range excess g(r) − 1. Fitting a cluster model to that excess gives the size directly,
with no eps / min_samples to calibrate. It also doesn't depend on labelling density, because g(r) is
thinning-invariant. This is the idea behind PC-PALM (Sengupta et al., *Nat. Methods* 2011).

**Model.** Each nanodomain is a Gaussian cloud of localizations with standard deviation σ₁ per axis. Domains
themselves form groups with spread σ_group. That's the same two-level "domains of domains" structure
`hmt.simulate.generate_nucleus` builds with Thomas seed placement. For that process

g(r) − 1 = A₁ exp(−r² / 4σ₁²) + A₂ exp(−r² / 4σ₂²),  with σ₂² = σ_group² + σ₁²

The first term comes from pairs within one domain, the second from pairs in different domains of the same
group. The fit runs over `BLINK_R_NM`–`SIZE_FIT_R_NM`. It starts from `hmt.measure.fit_thomas_parameters`,
the toolbox's one-scale version of the same model fitted as a straight line of ln(g − 1) against r². On these
cells a single scale leaves 2–5× larger residuals than two (compare `rms_one_scale` and `rms_two_scale`).
The fit runs on each subsample's g(r) separately, so every parameter gets a mean and a `CI_LEVEL` interval per
cell. Phases are then compared with one value per cell, as for the summary numbers in step 4.

| Column | What it is |
|---|---|
| `domain_diameter_nm` | 2 × the 75th-percentile radius of a Gaussian domain, 2σ√(2 ln 4) ≈ 3.33σ, after removing localization precision (σ² − σ_loc², σ_loc = median uncertainty). The same statistic as `geometric_diameter_2D` in `hmt.cluster.calculate_nanodomain_characteristics` and the DBSCAN calibration, so it compares directly with DBSCAN results |
| `domain_fwhm_nm` | full width at half maximum, 2.355σ, the convention in many SMLM papers |
| `group_sigma_nm` | spread of domain centres within a group, √(σ₂² − σ₁²) |
| `domain_density_per_um2` | κ = 1 / (4π A₁ σ₁²). *Apparent:* a uniform background fraction f dilutes A₁ by (1 − f)², so κ is overestimated by 1 / (1 − f)². The sizes are unaffected by background |

Limitations:
- The size assumes a Gaussian domain profile.
- With a spread of domain sizes, it's a pair-weighted average. A domain with n localizations contributes about
  n² pairs, so larger and denser domains count more than in a per-domain DBSCAN mean.

In [ ]:
P75_DIAMETER = 2 * np.sqrt(2 * np.log(4))  # 2 x 75th-percentile radius of a 2D Gaussian, in units of sigma
FWHM = 2 * np.sqrt(2 * np.log(2))


def gauss_excess(r, sigma, amp):
    # g(r) - 1 of Gaussian clusters with per-axis SD sigma (Thomas process)
    return amp * np.exp(-r ** 2 / (4 * sigma ** 2))


def two_scale_excess(r, s1, a1, s2, a2):
    return gauss_excess(r, s1, a1) + gauss_excess(r, s2, a2)


def fit_domain_scales(r, g, sigma_loc):
    """
    Fits one- and two-scale Gaussian cluster models to g(r) - 1 over BLINK_R_NM..SIZE_FIT_R_NM.
    Returns the domain size, group spread and apparent domain density from the two-scale fit, the
    rms residual of both fits, and the fitted parameters ("params" = s1, a1, s2, a2; "params_one").
    """
    sel = (r >= BLINK_R_NM) & (r <= SIZE_FIT_R_NM)
    r, y = r[sel], g[sel] - 1
    # starting point: the toolbox's linearised one-scale Thomas fit (ln(g - 1) against r^2)
    _, s0, ok = hmt.measure.fit_thomas_parameters({"r": r, "g": g[sel]}, BLINK_R_NM, 0.0, SIZE_FIT_R_NM)
    p1, _ = curve_fit(gauss_excess, r, y, p0=[s0 if ok else 100.0, max(y[0], 1e-3)], bounds=([1, 0], [1e4, np.inf]))
    try:
        p2, _ = curve_fit(two_scale_excess, r, y, p0=[0.6 * p1[0], y[0] / 2, 1.5 * p1[0], y[0] / 4],
                          bounds=([1, 0, 1, 0], [1e4, np.inf, 1e4, np.inf]))
    except RuntimeError:
        p2 = np.full(4, np.nan)
    s1, a1, s2, a2 = p2 if p2[0] <= p2[2] else p2[[2, 3, 0, 1]]  # term 1 = the narrower one

    sigma_domain = np.sqrt(max(s1 ** 2 - sigma_loc ** 2, 0.0))
    rms = lambda f, p: float(np.sqrt(np.mean((y - f(r, *p)) ** 2)))
    return {"domain_sigma_nm": sigma_domain,
            "domain_diameter_nm": P75_DIAMETER * sigma_domain,
            "domain_fwhm_nm": FWHM * sigma_domain,
            "group_sigma_nm": np.sqrt(max(s2 ** 2 - s1 ** 2, 0.0)),
            "domain_density_per_um2": 1e6 / (4 * np.pi * a1 * s1 ** 2),
            "rms_one_scale": rms(gauss_excess, p1),
            "rms_two_scale": rms(two_scale_excess, (s1, a1, s2, a2)),
            "params": (s1, a1, s2, a2), "params_one": tuple(p1)}


size_rows, size_fits = [], {}
for cell in nuclei:
    short, long = results[(cell, "short")]["data"], results[(cell, "long")]["data"]
    sigma_loc = float(nuclei[cell]["uncertainty [nm]"].median())
    for i in range(min(len(short["g"]), len(long["g"]))):
        r, g = join_g(short, long, i)
        fit = fit_domain_scales(r, g, sigma_loc)
        size_fits.setdefault(cell, []).append((r, g, fit))
        size_rows.append({"cell": cell, "draw": i, "sigma_loc_nm": sigma_loc,
                          **{k: v for k, v in fit.items() if not k.startswith("params")}})

size_draws = pd.DataFrame(size_rows).set_index(["cell", "draw"])
size_summary = summarise(size_draws)
n_failed = size_draws["domain_diameter_nm"].isna().groupby(level="cell").sum()
if n_failed.any():
    print(f"two-scale fit failed on some subsamples (left out of the means and intervals): "
          f"{n_failed[n_failed > 0].to_dict()}")
size_digits = {"domain_diameter_nm": 0, "domain_fwhm_nm": 0, "group_sigma_nm": 0, "domain_density_per_um2": 1}
size_phase_summary, size_phase_tests, size_phase_pairs = compare_phases(size_summary, list(size_digits))
display(ci_text(size_summary, {**size_digits, "rms_one_scale": 3, "rms_two_scale": 3}))
display(phase_text(size_phase_summary, size_digits))
display(size_phase_tests.round(4))
show_pairs(size_phase_pairs)

The fit for each cell. Points are the mean g(r) − 1 over the subsamples, with its 95% interval shaded (often
narrower than the points). The black line is the mean of the subsamples' two-scale fits, with their 95% interval
in grey, and the dashed lines are its two terms. Each title gives the cell and its domain diameter: the mean
and, in brackets, the 95% interval. If the black line misses the points in a systematic way, the Gaussian-domain model doesn't
describe that cell, and its size shouldn't be trusted.

In [ ]:
n_cols = min(4, len(size_fits))
n_rows = -(-len(size_fits) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 4.0 * n_rows), squeeze=False)
for ax in axes.flat[len(size_fits):]:
    ax.axis("off")
rr = np.linspace(BLINK_R_NM, SIZE_FIT_R_NM, 300)
for k, (ax, (cell, fits)) in enumerate(zip(axes.flat, size_fits.items())):
    r = fits[0][0]  # every subsample's g(r) is on the same r grid
    sel = r <= SIZE_FIT_R_NM
    g_mean, g_lo, g_hi = mean_ci(np.array([g[sel] - 1 for _, g, _ in fits]))
    params = np.array([fit["params"] for _, _, fit in fits])
    params = params[np.isfinite(params).all(1)]  # failed fits are NaN
    within = np.array([gauss_excess(rr, s1, a1) for s1, a1, _, _ in params])
    between = np.array([gauss_excess(rr, s2, a2) for _, _, s2, a2 in params])
    total_mean, total_lo, total_hi = mean_ci(within + between)

    c = PHASE_COLORS[phase_of[cell]]
    ax.fill_between(r[sel], g_lo, g_hi, color=c, alpha=0.3, lw=0)
    ax.plot(r[sel], g_mean, "o", ms=4, color=c, label="g(r) − 1")
    ax.plot(rr, within.mean(0), color=MUTED, lw=1.2, ls="--", label="within a domain")
    ax.plot(rr, between.mean(0), color=MUTED, lw=1.2, ls=":", label="between domains")
    ax.fill_between(rr, total_lo, total_hi, color=INK, alpha=0.15, lw=0)
    ax.plot(rr, total_mean, color=INK, lw=1.5, label="two-scale fit")
    ax.axhline(0, color=AXIS, lw=1)
    m, lo, hi = size_summary.loc[cell, ["domain_diameter_nm", "domain_diameter_nm_lo", "domain_diameter_nm_hi"]]
    style(ax, "r (nm)", "g(r) − 1" if k % n_cols == 0 else "", f"{cell}\ndiameter {m:.0f} nm ({lo:.0f}–{hi:.0f})")
    if k == 0:
        ax.legend(frameon=False, loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "nanodomain_size_fits.png", dpi=200)
plt.show()

### Check: does the fit recover a known domain size?

Builds two-level Gaussian domain patterns with known σ₁ (`SIZE_CHECK_SIGMAS_NM`) inside the first cell's mask:
- group centres placed uniformly in the mask with `hmt.simulate.add_noise_locs`
- domain centres scattered around each group centre (`SIZE_CHECK_GROUP_SIGMA_NM`)
- localizations around each domain centre, with localization error
- 30% uniform background

Each pattern goes through the same estimator and fit as the data. The fitted sizes should land close to the
true ones. If they don't at the size measured in the data, read that cell's size with caution.

In [ ]:
def simulate_domains(mk, n_points, sigma_domain, sigma_group, sigma_loc, locs_per_domain=25, domains_per_group=5,
                     background=0.3, seed=0):
    # Two-level Gaussian cluster process inside a nucleus mask: group centres uniform in the mask, domain centres
    # Gaussian(sigma_group) around them, localizations Gaussian(sigma_domain) + localization error around those,
    # plus a uniform background. Points that land outside the mask are dropped, as binarize_nucleus does.
    # Returns the points (column "domain" = true domain index, -1 = background) and the true domain centres.
    rng = np.random.default_rng(seed)
    np.random.seed(seed)  # add_noise_locs draws from NumPy's global random state
    uniform = lambda n: hmt.simulate.add_noise_locs(n, mk["bands"], np.zeros(1), mk["x_min"], mk["y_min"],
                                                    px_size=MASK_BIN_NM)[["x [nm]", "y [nm]"]].to_numpy()
    n_domains = int(n_points * (1 - background) / locs_per_domain)
    groups = uniform(max(1, n_domains // domains_per_group))
    centres = np.repeat(groups, rng.poisson(domains_per_group, len(groups)), axis=0)
    centres = centres + rng.normal(0, sigma_group, centres.shape)
    per_domain = rng.poisson(locs_per_domain, len(centres))
    locs = np.repeat(centres, per_domain, axis=0)
    locs = locs + rng.normal(0, np.hypot(sigma_domain, sigma_loc), locs.shape)
    background_pts = uniform(int(n_points * background))
    pts = np.vstack([locs, background_pts])
    domain = np.concatenate([np.repeat(np.arange(len(centres)), per_domain), np.full(len(background_pts), -1)])

    ix = np.floor((pts[:, 0] - mk["x_min"]) / MASK_BIN_NM).astype(int)
    iy = np.floor((pts[:, 1] - mk["y_min"]) / MASK_BIN_NM).astype(int)
    inside = (ix >= 0) & (ix < mk["mask"].shape[0]) & (iy >= 0) & (iy < mk["mask"].shape[1])
    inside[inside] = mk["mask"][ix[inside], iy[inside]]
    sim = pd.DataFrame(pts[inside], columns=["x [nm]", "y [nm]"]).assign(domain=domain[inside])
    return sim, centres


check_mask = masks[first]
check_sigma_loc = float(np.median([nucleus["uncertainty [nm]"].median() for nucleus in nuclei.values()]))
check_rows, check_runs = [], []
for k, sigma_true in enumerate(SIZE_CHECK_SIGMAS_NM):
    t0 = time.time()
    sim, sim_centres = simulate_domains(check_mask, SHORT["n"], sigma_true, SIZE_CHECK_GROUP_SIGMA_NM,
                                        check_sigma_loc, seed=RANDOM_SEED + k)
    sim_short = pcf_draws(lambda i: sim.sample(min(SHORT["n"], len(sim)), random_state=k), SHORT, 1, check_mask)
    sim_long = pcf_draws(lambda i: sim.sample(min(LONG["n"], len(sim)), random_state=k), LONG, 1, check_mask)
    r_sim, g_sim = join_g(sim_short, sim_long, 0)
    fit = fit_domain_scales(r_sim, g_sim, check_sigma_loc)
    check_runs.append({"sigma_true": sigma_true, "sim": sim, "centres": sim_centres, "r": r_sim, "g": g_sim,
                       "fit": fit})
    check_rows.append({"true_diameter_nm": P75_DIAMETER * sigma_true, "fit_diameter_nm": fit["domain_diameter_nm"],
                       "true_group_sigma_nm": SIZE_CHECK_GROUP_SIGMA_NM, "fit_group_sigma_nm": fit["group_sigma_nm"]})
    print(f"domain sigma {sigma_true:.0f} nm: {len(sim):,} simulated points, {time.time() - t0:.0f} s")
size_check = pd.DataFrame(check_rows)
size_check.round(1)

How to read the check figure:
- **Top row:** a 2 µm window of each simulated nucleus, centred on a group of domains. Domain localizations
  are orange, background is grey, and black circles mark each true domain's 75th-percentile diameter, the
  size the fit should return.
- **Bottom row:** that pattern's g(r) − 1, with the same two-scale fit used on the real cells.
- **Right:** fitted against true diameter. Points on the dashed line are recovered exactly. The blue band
  is the range of sizes fitted in the cells, 95% intervals included, which should fall inside the tested range.

In [ ]:
SIM_COLOR = "#eb6834"  # categorical orange, so simulated data can't be mistaken for one of the blue S phases
WINDOW_NM = 2000.0

# Each window is centred on the simulated domain nearest the deepest point of the mask (contour band 0),
# so it sits inside the nucleus and always shows a group of domains
core = np.argwhere(check_mask["bands"] == 0)[0]
core_xy = np.array([check_mask["x_min"], check_mask["y_min"]]) + (core + 0.5) * MASK_BIN_NM
half = WINDOW_NM / 2

n_runs = len(check_runs)
mosaic = [[f"pattern{k}" for k in range(n_runs)] + ["recovery"], [f"fit{k}" for k in range(n_runs)] + ["recovery"]]
fig, axd = plt.subplot_mosaic(mosaic, figsize=(4.1 * (n_runs + 1), 8.4))
rr = np.linspace(BLINK_R_NM, SIZE_FIT_R_NM, 300)
for k, run in enumerate(check_runs):
    true_d = P75_DIAMETER * run["sigma_true"]
    fit = run["fit"]

    # simulated pattern with the true domain sizes drawn on
    ax = axd[f"pattern{k}"]
    cx, cy = run["centres"][np.argmin(np.hypot(*(run["centres"] - core_xy).T))]
    sim = run["sim"]
    win = sim[sim["x [nm]"].between(cx - half, cx + half) & sim["y [nm]"].between(cy - half, cy + half)]
    bg = win["domain"].to_numpy() < 0
    ax.scatter((win["x [nm]"][bg] - cx) / 1e3, (win["y [nm]"][bg] - cy) / 1e3, s=4, color=CSR_COLOR, alpha=0.5,
               linewidths=0, label="background")
    ax.scatter((win["x [nm]"][~bg] - cx) / 1e3, (win["y [nm]"][~bg] - cy) / 1e3, s=4, color=SIM_COLOR,
               linewidths=0, label="domain localizations")
    near = np.all(np.abs(run["centres"] - [cx, cy]) < half + true_d, axis=1)
    for x0, y0 in run["centres"][near]:
        ax.add_patch(plt.Circle(((x0 - cx) / 1e3, (y0 - cy) / 1e3), true_d / 2e3, fill=False, color=INK, lw=0.7))
    ax.set_xlim(-half / 1e3, half / 1e3)
    ax.set_ylim(-half / 1e3, half / 1e3)
    ax.set_aspect("equal")
    style(ax, "x (µm)", "y (µm)" if k == 0 else "", f"True domain diameter {true_d:.0f} nm")
    ax.grid(False)
    if k == 0:
        ax.legend(frameon=True, framealpha=0.9, edgecolor="none", loc="upper left", fontsize=8, markerscale=3)

    # the same two-scale g(r) fit as on the real cells
    ax = axd[f"fit{k}"]
    s1, a1, s2, a2 = fit["params"]
    sel = run["r"] <= SIZE_FIT_R_NM
    ax.plot(run["r"][sel], run["g"][sel] - 1, "o", ms=4, color=SIM_COLOR, label="simulated g(r) − 1")
    ax.plot(rr, gauss_excess(rr, s1, a1), color=MUTED, lw=1.2, ls="--", label="within a domain")
    ax.plot(rr, gauss_excess(rr, s2, a2), color=MUTED, lw=1.2, ls=":", label="between domains")
    ax.plot(rr, two_scale_excess(rr, *fit["params"]), color=INK, lw=1.5, label="two-scale fit")
    ax.axhline(0, color=AXIS, lw=1)
    style(ax, "r (nm)", "g(r) − 1" if k == 0 else "", f"Fitted diameter {fit['domain_diameter_nm']:.0f} nm")
    if k == 0:
        ax.legend(frameon=False, loc="upper right", fontsize=8)

# fitted vs true, with the range measured in the real cells
ax = axd["recovery"]
lims = np.array([0.75 * size_check["true_diameter_nm"].min(), 1.15 * size_check["true_diameter_nm"].max()])
ax.axhspan(size_summary["domain_diameter_nm_lo"].min(), size_summary["domain_diameter_nm_hi"].max(),
           color=PHASE_COLORS["S2"], alpha=0.12, lw=0, label="range fitted in the cells")
ax.plot(lims, lims, color=AXIS, ls="--", lw=1.2, label="fitted = true")
ax.plot(size_check["true_diameter_nm"], size_check["fit_diameter_nm"], "o", ms=9, color=SIM_COLOR,
        label="simulated domains")
for true_d, fit_d in zip(size_check["true_diameter_nm"], size_check["fit_diameter_nm"]):
    ax.annotate(f"{100 * (fit_d - true_d) / true_d:+.0f}%", (true_d, fit_d), xytext=(8, -4),
                textcoords="offset points", color=INK_2, fontsize=9)
ax.set_xlim(*lims)
ax.set_ylim(*lims)
ax.set_aspect("equal")
style(ax, "True diameter, p75 (nm)", "Fitted diameter, p75 (nm)", "Recovery of known domain sizes")
ax.legend(frameon=False, loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "nanodomain_size_check.png", dpi=200)
plt.show()

### Nanodomain parameters per S phase

As for the summary numbers: each dot is a cell, with its subsample interval as a thin bar. The black line and bar
beside each phase are the mean over its cells and the 95% confidence interval of that mean. The x labels give the
Kruskal–Wallis and trend tests, and `size_phase_pairs` holds the pairwise tests.

In [ ]:
panels = [("domain_diameter_nm", "Nanodomain diameter, p75 (nm)"),
          ("group_sigma_nm", "Domain group spread (nm)"),
          ("domain_density_per_um2", "Apparent domain density (µm⁻²)")]
fig = phase_panels(size_summary, size_phase_summary, size_phase_tests, panels, width=3.4)
fig.savefig(OUTPUT_DIR / "nanodomain_size_summary.png", dpi=200)
plt.show()

size_draws.to_csv(OUTPUT_DIR / "nanodomain_size_per_draw.csv")
size_summary.to_csv(OUTPUT_DIR / "nanodomain_size.csv")
size_phase_summary.to_csv(OUTPUT_DIR / "nanodomain_size_phase_summary.csv")
size_phase_tests.to_csv(OUTPUT_DIR / "nanodomain_size_phase_tests.csv")
size_phase_pairs.to_csv(OUTPUT_DIR / "nanodomain_size_phase_pairwise.csv", index=False)
size_check.to_csv(OUTPUT_DIR / "nanodomain_size_check.csv", index=False)
print(f"nanodomain size outputs -> {OUTPUT_DIR}")